# Ensembles in practice

In the slides we saw the two ways to combine many decision trees:

- **Bagging / random forests**: deep trees fit independently, then averaged.
- **Boosting / gradient boosting**: shallow trees fit sequentially on the residuals.

Here we put both to work on a real regression dataset and **tune their hyperparameters** with randomized search. The conceptual differences and the meaning of each hyperparameter are covered in the slides; this notebook focuses on the code.

## Loading the data

We use the California housing dataset, where the goal is to predict the median house value of a district from numerical features.

In [1]:
from pathlib import Path

# pandas: https://pandas.pydata.org/docs/
import pandas as pd

# Path keeps the location OS-independent instead of concatenating strings
data_path = Path("../datasets/california_housing.csv")
df = pd.read_csv(data_path)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


We prepare the inputs and the target:

- `target` is the `median_house_value` column we want to predict.
- `data` is every other column, except the categorical `ocean_proximity` which we drop for now (encoding is covered in its own lesson).
- We rescale the target to thousands of dollars (k$) so the error is easier to read.

In [2]:
target = df["median_house_value"]
data = df.drop(columns=["median_house_value", "ocean_proximity"])

# express the target in thousands of dollars (k$)
target = target / 1000

data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462


In [3]:
# train_test_split: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
from sklearn.model_selection import train_test_split

data_train, data_test, target_train, target_test = train_test_split(
    data,
    target,
    random_state=0,  # reproducible split
)

## Random forest

We tune a `RandomForestRegressor` with a randomized search over the hyperparameters introduced in the slides:

- `max_features`: how many features each split may consider (controls how decorrelated the trees are).
- `max_leaf_nodes`: an upper bound on the tree size.
- `min_samples_leaf`: the minimum number of samples a leaf must hold.

We keep `n_estimators` at its default (100): for a random forest it just needs to be *large enough*. We score with the (negative) mean absolute error so that a lower error is better.

> For clarity we do not use nested cross-validation here; we only show the effect of the parameters on the validation folds.

In [4]:
# RandomizedSearchCV: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html
from sklearn.model_selection import RandomizedSearchCV

# RandomForestRegressor: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
from sklearn.ensemble import RandomForestRegressor

param_distributions = {
    "max_features": [1, 2, 3, 5, None],          # features tried per split
    "max_leaf_nodes": [10, 100, 1000, None],      # cap on tree size (None = unlimited)
    "min_samples_leaf": [1, 2, 5, 10, 20, 50, 100],  # smallest allowed leaf
}

search_cv = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=2),
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",  # higher (less negative) is better
    n_iter=10,                          # number of random combinations to try
    random_state=0,
)
search_cv.fit(data_train, target_train)

# turn the cross-validation results into a readable, error-sorted table
columns = [f"param_{name}" for name in param_distributions]
columns += ["mean_test_error", "std_test_error"]

rf_cv_results = pd.DataFrame(search_cv.cv_results_)
rf_cv_results["mean_test_error"] = -rf_cv_results["mean_test_score"]
rf_cv_results["std_test_error"] = rf_cv_results["std_test_score"]
rf_cv_results[columns].sort_values(by="mean_test_error")

,param_max_features,param_max_leaf_nodes,param_min_samples_leaf,mean_test_error,std_test_error
3,2,None,2,36.168226,0.642317
7,None,None,20,36.718213,1.018198
0,2,1000,10,39.403747,0.687972
8,None,100,10,39.673390,0.898529
4,5,100,2,40.245139,0.795355
6,None,1000,50,40.936789,0.943563
2,1,100,1,53.488342,0.794284
9,1,100,2,53.958735,0.710196
1,3,10,10,57.194317,0.875625
5,1,None,100,58.653830,0.977439


The search favours **large `max_leaf_nodes`** (deep trees) while **large `min_samples_leaf`** hurts performance — exactly what we expect for a random forest, where each tree should overfit and averaging cleans up.

After the search, the best estimator is automatically refit on the full training set, so we can score it directly on the held-out test set.

In [6]:
error = -search_cv.score(data_test, target_test)
print(f"On average, our random forest regressor makes an error of {error:.2f} k$")

On average, our random forest regressor makes an error of 35.32 k$


## Histogram gradient boosting

Now we tune a `HistGradientBoostingRegressor`. As noted in the slides, its hyperparameters are **coupled**, so we search over them jointly:

- `max_iter`: the number of boosting stages (trees).
- `max_leaf_nodes`: keep the trees **shallow** (weak learners).
- `learning_rate`: how much each tree contributes; drawn on a log scale because it spans orders of magnitude.

> In real projects, prefer fixing `max_iter` to a large value and relying on early stopping rather than tuning it directly.

In [7]:
# loguniform samples on a log scale, e.g. for the learning rate
from scipy.stats import loguniform

# HistGradientBoostingRegressor: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
from sklearn.ensemble import HistGradientBoostingRegressor

param_distributions = {
    "max_iter": [3, 10, 30, 100, 300, 1000],      # number of boosting stages
    "max_leaf_nodes": [2, 5, 10, 20, 50, 100],     # keep each tree shallow
    "learning_rate": loguniform(0.01, 1),          # shrinkage, sampled log-uniformly
}

search_cv = RandomizedSearchCV(
    HistGradientBoostingRegressor(),
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",
    n_iter=20,
    random_state=0,
)
search_cv.fit(data_train, target_train)

columns = [f"param_{name}" for name in param_distributions]
columns += ["mean_test_error", "std_test_error"]

hgbt_cv_results = pd.DataFrame(search_cv.cv_results_)
hgbt_cv_results["mean_test_error"] = -hgbt_cv_results["mean_test_score"]
hgbt_cv_results["std_test_error"] = hgbt_cv_results["std_test_score"]
hgbt_cv_results[columns].sort_values(by="mean_test_error")

,param_max_iter,param_max_leaf_nodes,param_learning_rate,mean_test_error,std_test_error
14,300,100,0.018640,31.813336,0.580397
6,300,20,0.047293,32.600696,0.369029
2,30,50,0.176656,33.370455,0.481298
13,300,10,0.297739,33.641457,0.728196
9,100,20,0.083745,33.775322,0.524914
19,100,10,0.215543,33.932409,0.388936
12,100,20,0.067503,34.498371,0.587193
16,300,5,0.059290,37.071982,0.734295
1,100,5,0.160519,37.462015,0.550256
0,1000,2,0.125207,42.535998,0.795998


Among the best models, a **smaller `learning_rate`** goes hand in hand with **more trees or larger trees**. Detailed conclusions are hard to draw because the best value of each hyperparameter depends on the others.

We again estimate the generalization performance on the held-out test set.

In [8]:
error = -search_cv.score(data_test, target_test)
print(f"On average, our HGBT regressor makes an error of {error:.2f} k$")

On average, our HGBT regressor makes an error of 31.29 k$


## Wrap-up

On this dataset the histogram gradient boosting model typically reaches a lower error than the tuned random forest, while predicting fast thanks to its shallow trees.